In [1]:
print(1111)

1111


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
apikey = os.getenv("OPENAI_API_KEY")
default_model = os.getenv("OPENAI_DEFAULT_MODEL")
g_apikey = os.getenv("GEMINI_API_KEY")
g_default_model = os.getenv("GEMINI_DEFAULT_MODEL")

if not apikey:
    raise RuntimeError("OPENAI_API_KEY is not set. Add it to the .env file.")
if not default_model:
    raise RuntimeError("OPENAI_DEFAULT_MODEL is not set. Add it to the .env file.")

default_model = default_model.strip().lower()
print(f"Using model: {default_model}")
print(f"Gemini API key configured: {bool(g_apikey)}")
print(f"Gemini model: {g_default_model}")

Using model: gpt-6-luna


In [2]:
from openai import OpenAI
client = OpenAI()

In [8]:
print(client.models.list())
for i in client.models.list().data:
    print(i.id)

SyncPage[Model](data=[Model(id='o1-2024-12-17', created=1734326976, object='model', owned_by='system', shutdown_date='2026-10-23'), Model(id='gpt-5', created=1754425777, object='model', owned_by='system', shutdown_date=None), Model(id='gpt-4o', created=1715367049, object='model', owned_by='system', shutdown_date=None), Model(id='gpt-4o-mini-transcribe', created=1742068596, object='model', owned_by='system', shutdown_date='2027-02-26'), Model(id='davinci-002', created=1692634301, object='model', owned_by='system', shutdown_date='2026-09-28'), Model(id='gpt-image-2-2026-04-21', created=1776399994, object='model', owned_by='system', shutdown_date=None), Model(id='gpt-4o-mini-tts-2025-12-15', created=1765610837, object='model', owned_by='system', shutdown_date=None), Model(id='gpt-image-1', created=1745517030, object='model', owned_by='system', shutdown_date='2026-10-23'), Model(id='gpt-5.2-chat-latest', created=1765344352, object='model', owned_by='system', shutdown_date='2026-08-10'), Mo

In [9]:
response = client.responses.create(
    model=default_model,
    input="바나나는 무슨 색깔이니?",
)

print(response.output_text)

보통 잘 익은 바나나는 노란색이에요. 덜 익으면 초록색이고, 너무 익으면 갈색 반점이 생길 수 있어요.


In [18]:
response = client.responses.create(
    model = default_model,
    instructions = '반말로 대답해',
    input = '바나나는 무슨 색깔이니?'
)
print(response.output_text)

보통 노란색이야. 덜 익으면 초록색이고, 너무 익으면 갈색 반점이 생겨.


In [21]:
response = client.responses.create(
    
    model = default_model,
    # instructions
    input = '줄바꿈 없이 숫자를 콤마로 구분해서 1부터 100까지 세어 주세요',
    max_output_tokens = 150
)
print(response.output_text)


1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,


In [10]:
%%time
response = client.responses.create(
    
    model = default_model,
    input = '줄바꿈 없이 숫자를 콤마로 구분해서 1부터 100까지 세어 주세요',
    instructions = '반말로 대답해',
    max_output_tokens = 150,
    stream = True
)

CPU times: total: 0 ns
Wall time: 503 ms


In [25]:
for event in response:
    if event.type == 'response.output_text.delta':
        print(event.delta, end="", flush=True)  

1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43

In [27]:
url = 'https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQlcRHHoboxHyMLMB8jADi2cWHqWvRf63xmOhqcYiYjGw&s=10'
response = client.responses.create(
    model=default_model,
    input=[{"role": "user",
            "content": [
                { "type": "input_text",
                  "text": "이 이미지의 핵심 내용을 한국어로 설명하고,학습 자료로 쓸 때 주의할 점을 한 가지 알려 줘.",
                },
                { "type": "input_image",
                  "image_url": url,
                },
            ],
        }
    ],
)
print(response.output_text)

산과 호수가 어우러진 풍경화입니다. 호숫가 마을과 배, 길을 따라 이동하는 사람들과 수레가 그려져 있어 자연 속 일상의 모습을 보여 줍니다.

**학습 자료로 쓸 때 주의할 점:** 그림은 예술가의 표현이므로, 특정 지역의 실제 풍경이나 당시 생활상을 정확히 기록한 자료라고 단정하지 마세요.


In [11]:
from pydantic import BaseModel, Field

class SentimentResult(BaseModel):
	label: str = Field(description="긍정, 부정, 중립 중 하나")
	confidence: int = Field(ge=0, le=100)
	reason: str
	
review = "배송은 빨랐지만 제품이 설명보다 작아서 아쉬웠어요."
response = client.responses.parse(
	model=default_model,
	input=[{"role": "system",
			"content": "리뷰 감정을 분석하세요. confidence는 0~100 정수입니다."
		},
		{"role": "user", "content": review},
	],
	text_format=SentimentResult,
)
result = response.output_parsed
print(result.label)
print(result.confidence)
print(result.reason)

부정
86
배송은 빨랐지만 제품이 설명보다 작아 아쉽다는 불만이 표현되었습니다.


In [ ]:
response = client.responses.create(
    model = default_model,
    input = '오늘 서울 날씨 알려줘',
    tools = [{'type':'web_search'}]
)
print(response.output_text)

In [13]:
from openai import OpenAI
import base64

client = OpenAI()

response = client.images.generate(
    model="gpt-image-2",
    prompt="""
        수업시간에 몰래 황새게임하는 이미지
    """,
    size="1024x1024"
)

image_base64 = response.data[0].b64_json

with open("step1.png", "wb") as f:
    f.write(base64.b64decode(image_base64))

print("저장 완료")

저장 완료


In [15]:
from openai import OpenAI
import base64

client = OpenAI()

# Step 1에서 생성한 이미지 불러오기
with open("step4.png", "rb") as image_file:

    response = client.images.edit(
        model="gpt-image-2",
        image=image_file,
        prompt="컴퓨터로 게임하는 거고 게임내용은 다리얇은 황새의 균형 잡는게임이야"
    )

# Base64 이미지 추출
image_base64 = response.data[0].b64_json

# PNG 파일로 저장
with open("step2.png", "wb") as f:
    f.write(base64.b64decode(image_base64))

print("step2.png 저장 완료")

step2.png 저장 완료


In [16]:
from openai import OpenAI
import base64

client = OpenAI()

with open("step2.png", "rb") as image_file:

    response = client.images.edit(
        model="gpt-image-2",
        image=image_file,
        prompt="""
다음 제목 텍스트를 추가하세요.

"수업시간에 게임하면 안돼!"

이미지 상단 중앙에 배치하세요.

텍스트는 크고 우아하며 읽기 쉽도록 만들어 주세요.
한글을 사용해 주세요.
        """
    )

image_base64 = response.data[0].b64_json

with open("step3.png", "wb") as f:
    f.write(base64.b64decode(image_base64))

print("step5.png 저장 완료")

step5.png 저장 완료


In [17]:
from openai import OpenAI

client = OpenAI()

text = """
수업시간에 게임하면 안돼!
"""

response = client.audio.speech.create(
    model="gpt-4o-mini-tts",
    voice="nova",
    input=text
)

with open("korean_voice.mp3", "wb") as f:
    f.write(response.read())

print("저장 완료")

저장 완료


In [ ]:
response = client.responses.create(
    model=default_model,
    input="안녕하세요",
)

print("사용량 정보:", response.usage)

In [4]:
from openai import OpenAI

client = OpenAI()
user_input = input("질문 : ").strip()

if user_input:
    response = client.chat.completions.create(
        model=default_model.strip().lower(),
        messages=[{"role": "user", "content": user_input}],
    )
    print("답변 :", response.choices[0].message.content or "")


답변 : 든든하게 먹자! 집이면 **김치볶음밥에 계란후라이**, 밖이면 **제육덮밥** 추천 😋
땡기는 게 매콤한 거야, 뜨끈한 거야?


In [ ]:
from openai import OpenAI

client = OpenAI()

while True:
    user_input = input("질문 : ").strip()

    if user_input.lower() in {"exit", "quit", "종료"}:
        print("대화를 종료합니다.")
        break
    if not user_input:
        continue

    response = client.chat.completions.create(
        model=default_model.strip().lower(),
        messages=[{"role": "user", "content": user_input}],
    )
    print("A :", response.choices[0].message.content or "")

A : 안녕하세요! 무엇을 도와드릴까요?
A : 오늘은 제육덮밥 먹자. 든든하고 실패 없지 😋
A : 제육 별로구나. 다른 메뉴로 골라보자—뭐 먹고 싶어?


In [ ]:
from openai import OpenAI

client = OpenAI()
messages = []

while True:
    user_input = input("질문 : ").strip()

    if user_input.lower() in {"exit", "quit", "종료"}:
        print("대화를 종료합니다.")
        break
    if not user_input:
        continue

    messages.append({"role": "user", "content": user_input})
    stream = client.chat.completions.create(
        model=default_model.strip().lower(),
        messages=messages,
        stream=True,
    )

    answer_parts = []
    print("답변 :", end=" ")
    for chunk in stream:
        content = chunk.choices[0].delta.content
        if content:
            print(content, end="", flush=True)
            answer_parts.append(content)
    print()

    messages.append({"role": "assistant", "content": "".join(answer_parts)})

A : 서울에서 유명한 전통음식점 3곳을 꼽으면 다음과 같습니다.

1. **우래옥** — 평양냉면·불고기 / 중구 주교동  
2. **하동관** — 곰탕 / 중구 명동  
3. **진주회관** — 콩국수 / 중구 서소문동
A : 어느 식당을 말씀하시는지 표시가 빠진 것 같아요. 우선 세 곳 모두 서울역에서 가는 방법을 알려드릴게요.

- **우래옥**: 지하철 4호선 서울역 → 동대문역사문화공원역에서 5호선 환승 → 을지로4가역 하차. 식당까지 도보 약 5~10분.
- **하동관**: 지하철 4호선 서울역 → 명동역 하차 후 도보 약 10분.
- **진주회관**: 지하철 1호선 서울역 → 시청역 하차 후 도보 약 5~10분.

소요 시간과 도보 경로는 출구와 열차 대기 시간에 따라 달라질 수 있어요.
대화를 종료합니다.


In [6]:
# 함께해보기 12-1

question = input("질문 : ").strip()

response = client.responses.create(
    model=default_model,
    instructions = '당신은 친절한 도우미입니다. 한국어로 대답해주세요',
    input=question,
)

print("답변 :", response.output_text)

답변 : 안녕하세요! 무엇을 도와드릴까요? 😊


In [ ]:
# 함께해보기 12-2


In [7]:
# 함께해보기 12-4

while True:
    question = input('질문하세요 : ').strip()
    if question.lower() in ['q', 'exit', 'quit']:
        break

    response = client.responses.create(
        model=default_model,
        instructions='당신은 친절한 도우미입니다. 한국어로 대답해주세요',
        input=question,
        stream=True
    )

    for event in response:
        if event.type == 'response.output_text.delta':
            print(event.delta, end='', flush=True)

안녕하세요! 무엇을 도와드릴까요?배고프구나! 😋 집에 있는 재료 말해주면 간단히 뭐 해 먹을지 같이 골라줄게. 아니면 지금 당기는 게 있어—밥, 면, 간식?어느 지역에서 찾으시나요? 원하시는 음식 종류나 예산도 알려주시면 딱 맞게 추천해드릴게요! 😊그럼 작은 상식 하나! 문어는 심장이 세 개예요. 두 개는 아가미로 피를 보내고, 나머지 하나는 온몸으로 피를 보낸답니다. 🐙

KeyboardInterrupt: Interrupted by user

In [8]:
id = None
while True:
    question = input('질문하세요 : ').strip()
    if question.lower() in ['q', 'exit', 'quit']:
        break

    request = {
        'model': default_model,
        'input': question  
    }          

    if id :
        request['previous_response_id'] = id
    
    response = client.responses.create(**request)
    id = response.id
    print("응답:", response.output_text)
    

응답: 안녕! 좋은 아침이야 ☀️ 오늘도 기분 좋은 하루 보내!

응답: 그냥 느낌으로는 오전 9시쯤? 😄 정확한 시간은 네 위치나 시계를 볼 수 없어서 몰라!

응답: 공원이 크게 있다면… 혹시 서울 올림픽공원 근처야? 😄 큰 공원은 많아서 아직은 찍기 어렵다! 힌트 하나만 더 줘봐.

응답: 그럼 서울 안이라고 치면… 서울숲이나 월드컵공원 근처? 😄 아니면 서울 밖일 수도 있겠다—공원이 바다나 산 근처야?
응답: 서울 올림픽공원은 아니라고 했어! 😄
응답: 그럼 내 추측은… 서울숲 근처! 맞아? 😄
응답: 오, 맞혔다! 😄 그럼 서울숲 근처에 있구나. 좋은 아침이야—서울숲 산책하기 딱 좋겠다!
응답: 응? ㅂ은 오타야? 😄
